## Ch9-01 -- Querying the model for what has, and has not, been claimed

This notebook introduces a real requirement-coverage report, built by joining every named `RequirementUsage` against every `SatisfyRequirementUsage` the model actually carries; after running it you can see which requirements have a real claim of satisfaction against them, which candidates were checked and with what polarity, and which have none at all.

Chapters 3 through 8 declared satisfy claims one at a time, each notebook adding or checking a single candidate against a single requirement. This chapter asks a different question across the whole model at once: for every requirement usage the model declares, has anyone actually claimed a candidate satisfies it, and of which polarity? This notebook adds no new model element: `models/ch08-cumulative.sysml`, the real, current cumulative model Chapter 8 committed, already has everything this query needs, so this chapter queries it directly rather than growing a `models/ch09-cumulative.sysml` file that would carry nothing new (a design choice recorded in this chapter's own [index.md](index.md)).

In [1]:
from pathlib import Path
import opensysml
from toaster.report import format_diagnostics

conn = opensysml.connect(version="v0.9.0")
source = Path("../../models/ch08-cumulative.sysml").read_text()
model = conn.load_from_content(source, strict=False)
assert model.ok, f"Model failed: {format_diagnostics(model.diagnostics)}"


The model loads cleanly. Before querying it, the same language-tier control every chapter carries: a `satisfy` naming a requirement that was never declared still fails to load, which matters directly for a chapter about what has and has not been claimed -- a broken reference can never silently count as a claim.

In [2]:
# A satisfy claim naming a requirement that doesn't exist fails to load; it can
# never silently show up as a "claim" this chapter's coverage query would count.
bad_source = """
package Bad {
    private import ScalarValues::*;
    part def Widget { attribute x : Real default = 1.0; }
    part w : Widget;
    assert satisfy missingReq by w;
}
"""
bad = conn.load_from_content(bad_source, strict=False)
assert not bad.ok, "Expected failure for an undeclared requirement"
print(f"Negative control ok: bad.ok={bad.ok}")


Negative control ok: bad.ok=False


With the model loaded, the coverage report starts from the same two surfaces every chapter's own satisfy claim already used: every named `RequirementUsage` from `model.query()`, and every `SatisfyRequirementUsage`, named or not, from `get_satisfy_relationships()` (D-001: `model.query()` returns none of these directly). Each raw satisfy element carries `subsets` (the requirement it claims against), `subject` (the candidate, when there is one), `isNegated` (a positive or a negative claim) and `sysx:declaredKeyword` (`"verify"` for a verification-case objective, which has no subject at all -- the same distinction `conformance.satisfaction_claims_evaluated()` already relies on, Chapter 8 notebook 02).

In [3]:
from toaster.query import find_requirements, get_satisfy_relationships, ApiIndex

idx = ApiIndex(model)
requirements = sorted(r.id for r in find_requirements(model))
raw_satisfies = get_satisfy_relationships(model)

claims = []
for s in raw_satisfies:
    claims.append({
        "id": s.get("qualifiedName"),
        "requirement": idx.qn(s["subsets"]) if "subsets" in s else None,
        "subject": idx.qn(s["subject"]) if "subject" in s else None,
        "is_verify": s.get("sysx:declaredKeyword") == "verify",
        "is_negated": bool(s.get("isNegated", False)),
    })

print(f"requirements: {requirements}")
for c in claims:
    print(c)


requirements: ['ToasterDemo::heatGenerationReq', 'ToasterDemo::timely']
{'id': 'ToasterDemo::slow::@1', 'requirement': 'ToasterDemo::timely', 'subject': 'ToasterDemo::slow', 'is_verify': False, 'is_negated': True}
{'id': 'ToasterDemo::TimelyToastTest::@2::@0', 'requirement': 'ToasterDemo::timely', 'subject': None, 'is_verify': True, 'is_negated': False}
{'id': 'ToasterDemo::rated::@1', 'requirement': 'ToasterDemo::heatGenerationReq', 'subject': 'ToasterDemo::rated', 'is_verify': False, 'is_negated': False}
{'id': 'ToasterDemo::weak::@1', 'requirement': 'ToasterDemo::heatGenerationReq', 'subject': 'ToasterDemo::weak', 'is_verify': False, 'is_negated': True}


Four claims total, over two requirements. Joining them by requirement, split by polarity and by whether the claim is a verification-case objective (which names a requirement but claims nothing about a subject), is the actual coverage report.

In [4]:
coverage = {r: {"positive": [], "negative": [], "verify_objectives": []} for r in requirements}
for c in claims:
    if c["requirement"] not in coverage:
        continue
    if c["is_verify"]:
        coverage[c["requirement"]]["verify_objectives"].append(c["id"])
    elif c["is_negated"]:
        coverage[c["requirement"]]["negative"].append(c["subject"])
    else:
        coverage[c["requirement"]]["positive"].append(c["subject"])

for r in requirements:
    cov = coverage[r]
    positive, negative, verify_objectives = cov["positive"], cov["negative"], cov["verify_objectives"]
    covered = bool(positive)
    print(f"{r}:")
    print(f"  positive satisfy claims: {positive}")
    print(f"  negative satisfy claims: {negative}")
    print(f"  verify objectives (no subject): {verify_objectives}")
    print(f"  covered (has >=1 positive claim): {covered}")

assert coverage["ToasterDemo::heatGenerationReq"]["positive"] == ["ToasterDemo::rated"]
assert coverage["ToasterDemo::heatGenerationReq"]["negative"] == ["ToasterDemo::weak"]
assert coverage["ToasterDemo::timely"]["positive"] == []


ToasterDemo::heatGenerationReq:
  positive satisfy claims: ['ToasterDemo::rated']
  negative satisfy claims: ['ToasterDemo::weak']
  verify objectives (no subject): []
  covered (has >=1 positive claim): True
ToasterDemo::timely:
  positive satisfy claims: []
  negative satisfy claims: ['ToasterDemo::slow']
  verify objectives (no subject): ['ToasterDemo::TimelyToastTest::@2::@0']
  covered (has >=1 positive claim): False


`heatGenerationReq` is covered on both sides: `rated` was checked and found to satisfy it, `weak` was checked and found not to. `timely` has never had a positive satisfy claim at all. The only claim against it is negative (`slow` does not satisfy it), and the only other reference is `TimelyToastTest`'s own `verify timely;` objective, which names the requirement but declares nothing about any subject -- it has no `subject` to join against, the exact reason `conformance.satisfaction_claims_evaluated()` skips it rather than reporting it as a finding (Chapter 8 notebook 02). This is a real, present gap in the tutorial's own accumulated model, not a scratch example built to fail on purpose: `nominal`, the usage meant to represent the toaster actually meeting its timing requirement, has no `assert satisfy timely by nominal` anywhere in `models/ch08-cumulative.sysml`. This does **not** mean `nominal` fails `timely`: `Toaster::cycleTime` is still a settable attribute, not derived from anything (unchanged since Chapter 2), so no one has ever actually checked whether `nominal` satisfies `timely` in the first place. The gap this query finds is an absence of a claim, not evidence of a failed one.

A second, independent surface confirms the same thing rather than taking this notebook's own join on faith: `conformance.satisfaction_claims_evaluated()` (Chapter 8 notebook 02) evaluates every satisfy claim the model actually has and reports no finding at all for `nominal` and `timely` together, because there is no such claim for it to evaluate, not because one was checked and passed.

In [5]:
from toaster import conformance

report = conformance.report(model, stage=(9, 1))
result = next(r for r in report["project"] if r.check_id == "satisfaction-claims-evaluated")
print(f"satisfaction-claims-evaluated: status={result.status}")
for f in result.findings:
    print(f"  finding: {f}")

nominal_timely_findings = [
    f for f in result.findings
    if f.get("subject") == "ToasterDemo::nominal" and f.get("requirement") == "ToasterDemo::timely"
]
assert nominal_timely_findings == [], "Expected no finding: there is no claim about nominal and timely to evaluate"
print()
print("No finding for nominal/timely: there was never a claim to evaluate, which is different from a claim that was evaluated and passed.")
conn.close()


satisfaction-claims-evaluated: status=passed

No finding for nominal/timely: there was never a claim to evaluate, which is different from a claim that was evaluated and passed.


The requirement usages and satisfy relationships declared in `models/ch08-cumulative.sysml`, queried above through two different surfaces (a direct join, and `conformance.report()`), produced the same coverage gap both times, confirming the finding is a property of the model itself, not an artifact of how it was asked.

Try the chapter exercise in `exercises/ch09/exercise.ipynb`: it asks you to produce a coverage table for the bread-handling requirements, using the query-and-join pattern this notebook builds.